# Zürich's drinking fountains from the Overpass Web API

Zürich has more than 1200 public drinking fountains. In this notebook we download them from OpenStreetMap using the Swiss [Overpass API](https://overpass.osm.ch/) and show them on an interactive map.

Tip: the query below can also be tested interactively on [overpass-turbo.eu](https://overpass-turbo.eu/).

## Libraries and settings

In [1]:
# Libraries
import os
import json
import time

import requests
import pandas as pd
import folium
from folium import plugins

# Current working directory
print(f'Current working directory: {os.getcwd()}')

Current working directory: /workspaces/data_ingestion/05_OverpassTurbo_WebAPI


## Get data

OSM tag [`amenity=drinking_water`](https://wiki.openstreetmap.org/wiki/Tag:amenity%3Ddrinking_water) in the city of Zürich. The data is saved to a file; when the notebook is run again, the data is loaded from this file (delete the file to download fresh data).

In [2]:
OVERPASS_URL = 'https://overpass.osm.ch/api/interpreter'
DATA_FILE = 'drinking_fountains.json'

query = """
[out:json][timeout:90];
area["name"="Zürich"]["admin_level"="8"]->.searchArea;
nwr["amenity"="drinking_water"](area.searchArea);
out center;
"""

if os.path.exists(DATA_FILE):
    with open(DATA_FILE) as f:
        data = json.load(f)
    print(f'Loaded {len(data)} elements from {DATA_FILE}')
else:
    # Web API request with retries (the Overpass server is often busy)
    max_attempts = 5
    wait_time = 10  # seconds
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.post(OVERPASS_URL, data={'data': query}, timeout=90)
            r.raise_for_status()
            data = r.json()['elements']
            print(f'Attempt {attempt}: success, {len(data)} elements received.')
            break
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f'Attempt {attempt} failed, server busy.')
            if attempt == max_attempts:
                raise RuntimeError('Overpass API not available.') from e
            print(f'Waiting {wait_time} seconds before retrying ...')
            time.sleep(wait_time)
            wait_time *= 2

    with open(DATA_FILE, 'w') as f:
        json.dump(data, f)

# Example of a single element
data[0]

Loaded 951 elements from drinking_fountains.json


{'type': 'node',
 'id': 60030906,
 'lat': 47.4135343,
 'lon': 8.5412781,
 'tags': {'amenity': 'drinking_water',
  'bottle': 'yes',
  'dog': 'yes',
  'drinking_water:description': 'Quellwasser',
  'fountain': 'bubbler',
  'material': 'metal',
  'name': 'Notwasserbrunnen',
  'operator': 'WVZ',
  'operator:wikidata': 'Q27229237',
  'ref': '6081',
  'wheelchair': 'yes'}}

In [3]:
# Flatten the nested JSON into a data frame
df_raw = pd.json_normalize(data)

# Ways have no lat/lon, but a center point
if 'center.lat' in df_raw:
    df_raw['lat'] = df_raw['lat'].fillna(df_raw['center.lat'])
    df_raw['lon'] = df_raw['lon'].fillna(df_raw['center.lon'])

# Select and rename relevant columns
columns = {
    'id': 'osm_id',
    'type': 'osm_type',
    'lat': 'lat',
    'lon': 'lon',
    'tags.name': 'name',
    'tags.drinking_water:description': 'water',
    'tags.fountain': 'fountain_type',
    'tags.material': 'material',
    'tags.bottle': 'bottle',
    'tags.dog': 'dog',
    'tags.wheelchair': 'wheelchair'}
df = df_raw.reindex(columns=columns.keys()).rename(columns=columns)

# Water source in English
df['water_source'] = (df['water']
                      .map({'Quellwasser': 'Spring water', 'Leitungswasser': 'Tap water'})
                      .fillna('Unknown'))

print(df.shape)
df.head()

(951, 12)


,osm_id,osm_type,lat,lon,name,water,fountain_type,material,bottle,dog,wheelchair,water_source
0,60030906,node,47.413534,8.541278,Notwasserbrunnen,Quellwasser,bubbler,metal,yes,yes,yes,Spring water
1,60733338,node,47.402830,8.499940,Tränkbrunnen Höngg,Leitungswasser,drinking,stone,yes,no,yes,Tap water
2,60733384,node,47.402164,8.499442,NaN,Leitungswasser,drinking,stone,yes,no,limited,Tap water
3,60734353,node,47.407291,8.550476,NaN,Quellwasser,bubbler,NaN,NaN,NaN,yes,Spring water
4,61263856,node,47.405301,8.503774,NaN,Leitungswasser,drinking,NaN,NaN,NaN,NaN,Tap water


In [4]:
# Number of fountains per water source
df['water_source'].value_counts()

water_source
Spring water    335
Tap water       318
Unknown         298
Name: count, dtype: int64

## Interactive fountain map

Use the layer control (top right) to switch between the swisstopo map and the aerial image, and to show or hide the layers. Click on a fountain for details.

In [5]:
# Colors per water source
colors = {'Spring water': '#2a78d6', 'Tap water': '#eb6834', 'Unknown': '#9a9994'}

# Swisstopo base maps (free to use, see https://www.geo.admin.ch/)
swisstopo = 'https://wmts.geo.admin.ch/1.0.0/{}/default/current/3857/{{z}}/{{x}}/{{y}}.{}'
attr = '&copy; <a href="https://www.swisstopo.admin.ch/">swisstopo</a>'

m = folium.Map(location=[df.lat.mean(), df.lon.mean()], zoom_start=13,
               tiles=None, control_scale=True)
folium.TileLayer(swisstopo.format('ch.swisstopo.swisstlm3d-karte-grau', 'png'),
                 attr=attr, name='swisstopo map').add_to(m)
folium.TileLayer(swisstopo.format('ch.swisstopo.swissimage', 'jpeg'),
                 attr=attr, name='swisstopo aerial image', show=False).add_to(m)

# One clustered layer per water source
for source, color in colors.items():
    subset = df[df['water_source'] == source]
    layer = folium.FeatureGroup(
        name=f'<span style="color:{color}">&#9679;</span> {source} ({len(subset)})')
    cluster = plugins.MarkerCluster(options={'disableClusteringAtZoom': 16}).add_to(layer)
    for _, row in subset.iterrows():
        details = row[['name', 'fountain_type', 'material', 'bottle', 'dog', 'wheelchair']]
        popup = '<br>'.join(f'<b>{k}:</b> {v}' for k, v in details.dropna().items())
        folium.CircleMarker(
            location=(row['lat'], row['lon']), radius=6,
            color='white', weight=1.5, fill=True, fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(f'<b>{source}</b><br>{popup}', max_width=250)).add_to(cluster)
    layer.add_to(m)

# Heatmap (switched off by default)
plugins.HeatMap(df[['lat', 'lon']].values.tolist(), name='Heatmap',
                radius=18, blur=15, show=False).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [6]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-09-29 20:57:41
Python Version: 3.12.14
-----------------------------------
